# Notebook 2: DataFrame Transformations, Complex Data (JSON) & Spark SQL (Student Lab)
### Hands-on Workshop: Apache Spark Foundation & Ingestion Framework (Day 1 Afternoon)
### Related Presentation Slides: Slides 14 - 16 (Module 4) and Slide 20 (Spark SQL)

---

## Learning Objectives:
1. Master nested structures: `struct` extraction via dot notation and array unnesting with `explode()` (Slide 15).
2. Relational column expressions: `F.col()`, SQL `CASE WHEN` equivalent (`F.when().otherwise()`) (Slide 14).
3. Aggregations and Window Functions (`Window.partitionBy()`).
4. Execute ANSI SQL directly against Spark DataFrames using Temp Views (`createOrReplaceTempView`) (Slide 20).


In [ ]:
# Setup environment & SparkSession
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("02_Transformations_and_SQL") \
    .master("local[*]") \
    .getOrCreate()

# Load raw events dataset
df_raw = spark.read.json("data/raw/bundesliga_events.json")
print("Data loaded. Schema preview:")
df_raw.printSchema()


---
## Step 1: Handling Nested JSON Structures — struct & array (Related: Slide 15)

In DWH, JSON events often arrive with nested objects:
* To extract a field from a `struct`: use `F.col("struct_name.field_name")`.
* To handle missing/null values: use `F.coalesce()` or `F.when().otherwise()`.


In [ ]:
# Flattening the nested JSON (StatsBomb struct dot-notation)
df_flattened = df_raw.select(
    F.coalesce(F.col("event_id"), F.col("id")).alias("event_id"),
    F.col("minute"),
    F.col("team.name").alias("team_name"),
    F.col("player.name").alias("player_name"),
    F.coalesce(F.col("position.name"), F.col("player.position")).alias("player_position"),
    F.col("type.name").alias("event_type"),
    F.col("shot.statsbomb_xg").alias("xg_value"),
    F.coalesce(F.col("shot.outcome.name"), F.col("shot.outcome")).alias("shot_outcome")
)

df_flattened.show(5)


---
## Step 2: SQL CASE WHEN Equivalent — F.when().otherwise() (Related: Slide 14)

SQL:
```sql
CASE 
    WHEN minute <= 45 THEN '1st_Half' 
    ELSE '2nd_Half' 
END AS match_period
```
PySpark:
```python
F.when(F.col("minute") <= 45, "1st_Half").otherwise("2nd_Half")
```


In [ ]:
df_categorized = df_flattened.withColumn(
    "match_period",
    F.when(F.col("minute") <= 45, "1st_Half").otherwise("2nd_Half")
).withColumn(
    "is_goal",
    F.when(F.col("shot_outcome") == "Goal", 1).otherwise(0)
)

df_categorized.select("player_name", "minute", "match_period", "shot_outcome", "is_goal").show(10)


---
## Step 3: Analytical Window Functions

In SQL DWH, window functions are standard tools (`ROW_NUMBER() OVER(PARTITION BY ... ORDER BY ...)`).
Let's rank players by total shots within each team:


In [ ]:
# 1. Calculate player total shots
df_player_shots = df_categorized.filter(F.col("event_type") == "Shot") \
    .groupBy("team_name", "player_name") \
    .agg(
        F.count("event_id").alias("total_shots"),
        F.sum("is_goal").alias("total_goals"),
        F.round(F.sum("xg_value"), 2).alias("total_xg")
    )

# 2. Define Window Specification
window_team = Window.partitionBy("team_name").orderBy(F.desc("total_shots"))

# 3. Apply ranking
df_ranked = df_player_shots.withColumn("team_rank", F.dense_rank().over(window_team))

df_ranked.show(20, truncate=False)


---
## Step 4: Spark SQL & Temp Views (Related: Slide 20)

You can run standard ANSI SQL directly on your DataFrames without moving data:


In [ ]:
# Register DataFrame as a temporary SQL view (Slide 20)
df_categorized.createOrReplaceTempView("v_events")

# Run native SQL
sql_query = """
SELECT 
    team_name,
    COUNT(CASE WHEN event_type = 'Shot' THEN 1 END) AS shots_count,
    SUM(is_goal) AS goals_scored,
    ROUND(SUM(xg_value), 2) AS team_total_xg,
    ROUND(SUM(is_goal) / NULLIF(COUNT(CASE WHEN event_type = 'Shot' THEN 1 END), 0) * 100, 1) AS shot_conversion_pct
FROM v_events
GROUP BY team_name
ORDER BY goals_scored DESC
"""

df_team_summary = spark.sql(sql_query)
df_team_summary.show()


---
## Hands-on Lab Exercise 2 (Corresponding to Module 4 Hands-on)

### Task:
Find the top 3 most clinical players in the dataset (players with the highest goals-to-shots ratio among those who had at least 3 shots).
Filter `total_shots >= 3`, calculate `conversion_rate = total_goals / total_shots`, and sort descending.


In [ ]:
# TODO: Write your solution below:
# Hint: Use df_player_shots, filter total_shots >= 3, add conversion_rate column

# df_clinical = ...

# Verification:
# df_clinical.show(5)
